In [1]:
import html
import re

from dotenv import load_dotenv
from IPython.display import display, HTML
from langchain_dartmouth.llms import ChatDartmouth
from langchain_core.prompts import ChatPromptTemplate
from tqdm.notebook import tqdm

from analysis_helpers import Participant
from analysis_helpers.constants import RAW_DIR

In [2]:
participants = Participant.load_all()
load_dotenv()

OUTPUT_DIR = RAW_DIR.joinpath('transcriptions-formatted')
OUTPUT_DIR.mkdir(exist_ok=True)

In [3]:
MODEL_NAME = 'qwen.qwen3.5-122b'

PRIMARY_SYSTEM_PROMPT = """\
You are preparing a transcript of spoken memory recall for embedding-based semantic analysis. A participant verbally recalled events from a TV episode (Atlanta, S1E1). You receive a raw transcript of their recall with little to no punctuation or capitalization. Your goal is to produce well-formed written sentences that faithfully preserve the participant's words. When the rules below conflict with each other, prioritize preserving the participant's actual word choices, grammar, and content over producing polished prose. This priority applies to what the participant said — the words themselves — not to how the text is structured. Apply the formatting rules (punctuation, capitalization, sentence breaks, stutter collapse) confidently; preserve the underlying word sequence faithfully.

Rules:
- Add proper punctuation (such as . , ? ! " ' —) and capitalization.
- Capitalize proper nouns including places, titles, and character names: Earn (Earnest), Alfred, Paper Boy, Darius, Van (Vanessa), Lottie, Dave, Kyle, KP, Prince, Uncle Riley, Loretta, and any other names should always be capitalized.
- Every sentence must end in a sentence-final punctuation mark (. ? or !). Do not use ellipses; complete trailing-off thoughts at the natural stopping point with a period.
- Remove clear non-word disfluencies ("um", "uh", "ah", "huh", "hmm", and similar). Examples:
    - and then hmm I think van turned on the news → And then I think Van turned on the news.
    - and earn's like uh I have no idea → And Earn's like, "I have no idea."
- Collapse stuttered verbatim and near-verbatim repetitions. Examples:
    - he opened the the door → He opened the door.
    - I I I think that's right → I think that's right.
    - then we see the guy who was the guy who was on the bus → Then we see the guy who was on the bus.
    - and that in that dream in the dream he remembers like being in a pool → and that in the dream, he remembers like being in a pool
    - and then paper boy earn and paper boy earn and darius were all looking at him like nodding like oh yeah like what a story like that's pathetic → And then Paper Boy, Earn, and Darius were all looking at him like nodding, like "Oh yeah, like what a story, like that's pathetic."
- Add quotation marks around the reported speech. When the participant uses "said", "went", "was like", or similar to introduce a character's speech, keep the introducing word. Examples:
    - Earn was like yeah of course → Earn was like, "Yeah, of course."
    - and she goes i don't know → And she goes, "I don't know."
    - he said get out of here → He said, "Get out of here."
- Distinguish reported speech (the character's actual words) from narrator commentary that follows words like "said." If what follows "said" is the participant describing the character's actions or self-correcting rather than the character's spoken words, do NOT use quotation marks. Examples:
    - she said she had a date that night → She said she had a date that night.
    - he said no I didn't and then he said but he actually did → He said, "No, I didn't," and then he said— but he actually did.
- Preserve self-corrections and restarts verbatim, marking the break with an em-dash. Examples: 
    - the guy went to I mean the woman went to the store → The guy went to— I mean the woman went to the store.
    - and then earn was like or alfred paper boy says oh we're late → And then Earn was like— or Alfred Paper Boy says, "Oh, we're late."
- The input transcript reflects casual unrehearsed speech, so self-corrections are common and often occur without an explicit restart marker like "I mean" or "or rather". Watch for cases where the participant starts a clause, abandons it, and starts a different clause that occupies the same syntactic slot. Mark these with an em-dash. Examples:
    - earn was there and earn's the woman that earn has a kid with her name is van → Earn was there, and Earn's— the woman that Earn has a kid with, her name is Van.
    - and then she said and then earn she said so you didn't make out with her → And then she said— and then earn— she said, "So you didn't make out with her?"
    - but then later they were chilling in the car listening to paper boy on his name is as an artist is as paper boy but the song's name is also paper boy → But then later they were chilling in the car listening to Paper Boy on— his name is as an artist is as Paper Boy, but the song's name is also Paper Boy.
- Do not silently fix other grammar errors: don't correct or improve tense, agreement, or aspect. Preserve non-standard grammatical constructions verbatim, using an em-dash for cadence breaks. Examples:
    - they was working at the airport → They was working at the airport.
    - it was him and his friend who on his name tag it said swiff → It was him and his friend who— on his name tag it said Swiff.
- Preserve malapropisms and "wrong" word choices verbatim. If the participant says a word that sounds like a mistake (e.g., "irregardless", "for all intensive purposes", etc.), keep it exactly as written. Do not silently substitute a more standard word.
- Preserve asides and self-directed speech, even if the content does not describe the episode. Examples:
  - her name was oh man I really don't remember this is hard I think her name was Fran → Her name was— oh man I really don't remember, this is hard. I think her name was Fran.
  - there was we didn't know the characters at the time but there was earn there there was Alfred → There was— we didn't know the characters at the time, but there was Earn, there was Alfred.
  - okay and then the next scene let me think for a second just trying to put it in chronological order → Okay, and then the next scene— let me think for a second, just trying to put it in chronological order.
- Do not remove conjunctions ("and", "or", etc.) when splitting long run-ons into multiple sentences; keep the conjunction as the first word of the next sentence.
- Do not paraphrase, summarize, reorder words, correct word choice, or add details not present in the input.
- Do not remove filler words: retain uses of "like", "kind of", "sorta", "I think", "you know", and similar.
- Do not add words to improve flow, smooth abrupt transitions, or fill in apparent omissions in the participant's speech.
- Do not add newlines or line breaks; do not split the text into paragraphs.
- Do NOT correct factual errors about the episode. If the participant misremembered something, keep their version of events.
- Output ONLY the cleaned transcript. No commentary, headers, or explanations.
- When in doubt about whether to substitute, add, or remove a word, default to preserving the input words verbatim. The formatting rules above (punctuation, capitalization, sentence segmentation, stutter collapse, restart-marking) should be applied confidently — these don't change what the participant said, only how it's written.
"""

LIKE_REMOVAL_SYSTEM_PROMPT = """\
You are removing filler uses of "like" from a cleaned transcript of spoken memory recall. The text has already been punctuated, capitalized, and structured. Your only job is to remove instances of "like" that function as filler, hedge, approximative, or discourse marker, while keeping instances that carry meaning.

General test for removal: read the sentence with "like" removed. If the result is grammatical and the meaning is preserved, "like" was filler — remove it. If removing "like" produces an ungrammatical or meaningless sentence, keep it.

Remove "like" in these positions:
- At the beginning of a sentence: "Like they seemed to kind of know each other, but not really well." → "They seemed to kind of know each other, but not really well."
- Between an article/possessive/preposition and a noun phrase: "drinking out of like a paper bag" → "drinking out of a paper bag"
- Before or within a verb phrase: "she was like making signs" → "she was making signs"
- At the start of a clause as a hedge: "like maybe he had to swim above the hands" → "maybe he had to swim above the hands"
- Between a copula and an adjective/noun: "he was like a rapper" → "he was a rapper"
- Repeated stutters of "like like": "she's like like, 'What'd she look like?'" → "she's like, 'What'd she look like?'"

Keep "like" in these positions:
- Introducing reported speech: "she was like, 'I don't know'" — keep
- As a comparison word meaning "similar to": "seaweed that looked like hands" — keep
- As a verb meaning "to enjoy": "I like him" / "he likes Flo Rida" — keep

Pay special attention to whether a use of "like" introduces reported speech versus serves as filler. Never remove "like" immediately preceding quote-enclosed speech UNLESS another word ("said", "went", "asked", or similar) already functions to introduce the speech. Examples:
- "And then Earn was like, 'Wait a scond.'" — "was like" introduces speech. Keep it.
- "So then he was like, 'No, that's not true at all.'" — "was like" introduces speech. Keep it.
- "And she was laughing at him being sort of like, 'What do you mean?'" — "being ... like" introduces speech. Keep it.
- "And Alfred said like, 'Get over here.'" — "said" introduces the speech; "like" is just filler. Remove it.
- "He was asking like, 'Are you sure?'" — "was asking" introduces the speech; "like" is just filler. Remove it.

When uncertain, default to removal. Do not change anything else about the text — preserve all other words, punctuation, capitalization, and sentence breaks exactly as given. Do not paraphrase, reorder, or add words.

Output ONLY the cleaned transcript. No commentary, headers, or explanations.\
"""

In [4]:
llm = ChatDartmouth(
    model_name=MODEL_NAME,
    temperature=0,
    max_tokens=20_000,
    seed=42,
    streaming=True,
    model_kwargs={"extra_body": {"chat_template_kwargs": {"enable_thinking": False}}}
)

primary_prompt = ChatPromptTemplate.from_messages([
    ('system', PRIMARY_SYSTEM_PROMPT),
    ('human', '{transcript}'),
])

like_removal_prompt = ChatPromptTemplate.from_messages([
    ('system', LIKE_REMOVAL_SYSTEM_PROMPT),
    ('human', '{transcript}'),
])

primary_chain = primary_prompt | llm

like_removal_chain = like_removal_prompt | llm

/opt/conda/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3699: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [5]:
pbar = tqdm(total=len(participants)*2, leave=False)
stream_display = display(HTML('<pre></pre>'), display_id=True)

for p in participants:
    for rectype in ('atlep1', 'delayed'):
        p_session_id = p.ses1_id if rectype == 'atlep1' else p.ses2_id
        p_output_dir = OUTPUT_DIR.joinpath(p.subid, p_session_id)
        p_output_dir.mkdir(parents=True, exist_ok=True)
        file_suffix = 'recall' if rectype == 'atlep1' else rectype
        output_fpath = p_output_dir.joinpath(f'{p_session_id}-{file_suffix}.txt')
        if output_fpath.is_file():
            pbar.update()
            continue

        transcript = p.transcripts[rectype]
        # ensure clean whitespace
        transcript = re.sub(r'\s+', ' ', transcript).strip()
        
        stream_header = f'=== {p.subid} / {rectype} (primary formatting) ===\n\n'
        chunks = []
        
        for chunk in primary_chain.stream({'transcript': transcript}):
            chunks.append(chunk.content)
            text = stream_header + ''.join(chunks)
            stream_display.update(HTML(f'<pre>{html.escape(text)}</pre>'))
        
        formatted_transcript = ''.join(chunks)
        
        if not formatted_transcript.strip():
            raise RuntimeError(f'Empty response for {p.subid}/{rectype}')
        
        stream_header = f'=== {p.subid} / {rectype} (like-removal) ===\n\n'
        chunks = []
        for chunk in like_removal_chain.stream({'transcript': formatted_transcript}):
            chunks.append(chunk.content)
            text = stream_header + ''.join(chunks)
            stream_display.update(HTML(f'<pre>{html.escape(text)}</pre>'))
            
        final_transcript = ''.join(chunks)
            
        output_fpath.write_text(final_transcript)
        stream_display.update(HTML('<pre></pre>'))
        pbar.update()
        
pbar.close()

  0%|          | 0/114 [00:00<?, ?it/s]